In [1]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from tinyshift.modelling import fourier_seasonality
from utilsforecast.preprocessing import fill_gaps
from mlforecast import MLForecast
from quantile_forest import RandomForestQuantileRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from mlforecast import MLForecast
from tinyconformal.series import ConformalQuantileTimeSeriesRegressor
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression


In [2]:
# Gerando dados em painel sintéticos para 5 séries temporais
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
df = fourier_seasonality(df, "ds", seasonality=["monthly"])
days_obsoletes=180
horizon = 12
train = df[:-horizon]
test = df[-horizon:]

# RandomForestQuantileRegressor

In [3]:

class QuantileRF(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        quantile=0.5,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
        **kwargs,
    ):
        self.quantile = quantile
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.kwargs = kwargs

    def fit(self, X, y):
        self.model_ = RandomForestQuantileRegressor(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            **self.kwargs,
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X, quantiles=self.quantile)

In [ ]:
def model_callable():
    models = {
        "RF-lo-90": QuantileRF(
            quantile=0.05, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-hi-90": QuantileRF(
            quantile=0.95, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-50": QuantileRF(
            quantile=0.50, n_estimators=100, max_depth=8, random_state=42
        ),
    }
    return MLForecast(
        models=models,
        lags=[1, 2, 7],
        freq="MS",
    )


models = model_callable()

cqr = ConformalQuantileTimeSeriesRegressor(
     learner=models,
     horizon=12,
     quantile_cols=[
         ("RF-lo-90", "RF-hi-90"),
     ],
     n_windows=7,
     alpha=0.10,
 )
cqr.fit(train, static_features=[])

In [20]:
cqr.predict_interval(h=7, X_df=test)

,unique_id,ds,RF-lo-90,RF-hi-90,RF-50
0,1,1960-01-01,297.70,427.00,362.0
1,1,1960-02-01,299.00,422.00,396.0
2,1,1960-03-01,233.05,591.95,405.0
3,1,1960-04-01,254.00,564.00,396.0
4,1,1960-05-01,246.00,546.00,420.0
5,1,1960-06-01,215.90,572.00,472.0
6,1,1960-07-01,141.00,621.00,548.0


In [31]:
cqr.evaluate(test, h=12, alpha=0.15)

,model,level,alpha,coverage_rate,interval_width_mean,mwis
1,RF,85%,0.15,0.917,342.792,343.903
0,RF,90%,0.15,0.750,323.617,456.950
